## 1. Research objective
The main objective of this project is to understand what each row in a price download actually represents before computing a return. Even daily equity data contains choices about market calendar, corporate-action adjustments, time zones, missing sessions and data correcitions. Before fitting any models we must establish which fields are adjusted, what information is lost when trades become daily bars whether gaps represent missing observations or legitimate non-trading days.

## 2. What a daily OHLCV bar represents
For a fixed interval, such as one trading day:
- Open is the first qualifying transaction price in the interval
- Close is the final qualifying transaction price in the interval or official session close under the vendor convention
- Low is the lowest qualifying transaction price
- High is the highest qualifying transaction price
- Volume is the number of shares traded under the vendor's aggregation rules

A row is often called a bar or a candle. Vendors may differ in how they treat auctions, corrections, time zones and corporate actions. We should never assume two vendors' fields are identical merely because the column names match.
A daily close is a reference observation, not a guarantee that a real order could have executed at that price.
A daily bar does not preserve the sequence of trades within the session. It does not reveal the bid–ask spread, available depth, trading venue, whether trades occurred at the bid or ask, or the price at which a real order of a given size could have executed.

## 3. Data sources and retrieval conventions
Two main data sources were used: yfinance and Alpha Vantage.

yfinance provides access to the full available daily history. Dividend and split records are also available. It was requested with `actions=True` and supplies `Adj Close`.
Setting `auto_adjust=False` prevents dividend-based adjustment to the price columns, but the vendor-supplied historical OHLC series is split-adjusted. 
Consequently, old yfinance OHLC values are not necessarily prices that were historically quoted at the exchange. For this reason `*_raw` means unchanged by this project, rather than indicating raw, unadjusted exchange quotations. Current-session rows may be returned before the session closes and are audited separately.

From Alpha Vantage we use the free `TIME_SERIES_DAILY` raw endpoint. Setting `outputsize="compact"` returns 100 most recent observations, which provide OHLCV but not adjusted close, dividend, or split factor information. These unavailable fields are stored as `pd.NA`. Because the free API enforces a strict burst limit, requests are paced. This compact history is used strictly for cross-vendor validation rather than as a primary historical source.

The New York Stock Exchange (XNYS) is the canonical exchange calendar. Automated gap auditing starts at `1970-01-01` because earlier historical calendar rules were not sufficiently reliable in the chosen package.

## 4. Security universe
| Asset | Role in the sample | Reason included |
|---|---|---|
| SPY | Broad-market ETF | Provides a highly liquid broad-market ETF for comparing ETF and single-stock data behavior |
| AAPL | Liquid mega-cap / split history | Tests high-volume observations and a well-documented split history over a long sample |
| MSFT | Liquid mega-cap / dividend payer | Provides long coverage, regular dividends, and a liquid large-cap comparison |
| KO | Long-history dividend payer | Tests very long historical coverage and exposes limitations in early exchange-calendar data |
| NVDA | Split history / volatile growth stock | Combines multiple splits with relatively high volatility and volume |
| TSLA | Volatile stock / split history | Tests volatile price and volume behavior alongside recent split history |
| FIZZ | Lower-volume stock | Provides a lower-volume security for auditing liquidity-sensitive vendor differences |
| PLTR | Shorter listing history | Tests shorter listing history and verifies that pre-listing dates are not treated as missing rows |
| COIN | Shorter listing history / high volatility | Combines shorter listing history with relatively high volatility and sensitivity to crypto-market conditions |
| BRK.B | Share-class and vendor-symbol case | Tests share-class symbol normalization, particularly `BRK.B` versus `BRK-B` |

This sample was selected to create varied data behavior, not because these securities were expected to outperform.

## 5. Canonical schema and index contract
Each normalized vendor table uses a two-level index `(date, asset_id)`. The keys must be unique and sorted first by `date`, then by `asset_id`. Date represents the U.S. regular trading-session date and is stored in a timezone-naive calendar date after normalization at nanosecond precision.
Vendor-specific ticker symbols are stored separately and are not used as the primary key. Both vendors share the same 11-column schema. Unavailable Alpha Vantage fields remain missing rather than being replaced with zero.

In [1]:
import sys
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parent
sys.path.append(str(ROOT / "src"))

from cleanbars.normalize import validate_panel

yf_panel = pd.read_parquet(ROOT / "data/processed/yfinance_daily_complete.parquet")
av_panel = pd.read_parquet(ROOT / "data/processed/alpha_vantage_daily_complete.parquet")

validate_panel(yf_panel)
validate_panel(av_panel)

schema_table = pd.DataFrame(
    {
        "column": yf_panel.columns,
        "yfinance_dtype": yf_panel.dtypes.astype(str).to_numpy(),
        "alpha_vantage_dtype": av_panel.dtypes.astype(str).to_numpy(),
    }
)

schema_table

,column,yfinance_dtype,alpha_vantage_dtype
0,vendor_symbol,string,string
1,open_raw,Float64,Float64
2,high_raw,Float64,Float64
3,low_raw,Float64,Float64
4,close_raw,Float64,Float64
5,volume_raw,Int64,Int64
6,adj_close,Float64,Float64
7,cash_dividend,Float64,Float64
8,split_factor,Float64,Float64
9,source,string,string


Both normalized vendor panels use the same canonical column order and pandas dtypes. This consistency allows validation and comparison functions to operate on either vendor without vendor-specific branching. The identical schema does not imply identical field definitions or coverage, which remain documented separately.

## 6. Coverage by asset and vendor
The final dataset shows clear differencess between two vendors. yfinance provides long, asset-dependent histories while Alpha Vantage is limited to exactly 100 most recent observations per asset. Shorter listing histories for PLTR and COIN are expected and are not considered missing-data failures. The finalized yfinance panel ends one day earlier than the original interim panel because partial 2026-07-17 rows were excluded.

In [2]:
from cleanbars.calendars import (
    build_session_completeness_audit, 
    filter_complete_sessions,
    build_calendar_audit,
    expected_sessions
)
from cleanbars.validate import (
    build_vendor_comparison,
    build_structural_checks,
    summarize_checks
)


yf_audit = build_session_completeness_audit(yf_panel)
av_audit = build_session_completeness_audit(av_panel)


yf_complete = filter_complete_sessions(yf_panel, yf_audit)
av_complete = filter_complete_sessions(av_panel, av_audit)

yf_cov = yf_complete.reset_index().groupby("asset_id")["date"].agg(
    first_date="min",
    last_date="max",
    observations="count"
)
yf_cov.insert(0, "vendor", "yfinance")

av_cov = av_complete.reset_index().groupby("asset_id")["date"].agg(
    first_date="min",
    last_date="max",
    observations="count"
)
av_cov.insert(0, "vendor", "alpha_vantage")

coverage_table = pd.concat([yf_cov, av_cov]).sort_values(["asset_id", "vendor"])

coverage_table

,vendor,first_date,last_date,observations
asset_id,,,,
AAPL,alpha_vantage,2026-02-23,2026-07-16,100
AAPL,yfinance,1980-12-12,2026-07-16,11489
BRK.B,alpha_vantage,2026-02-23,2026-07-16,100
BRK.B,yfinance,1996-05-09,2026-07-16,7594
COIN,alpha_vantage,2026-02-23,2026-07-16,100
COIN,yfinance,2021-04-14,2026-07-16,1320
FIZZ,alpha_vantage,2026-02-23,2026-07-16,100
FIZZ,yfinance,1991-09-13,2026-07-16,8771
KO,alpha_vantage,2026-02-23,2026-07-16,100


## 7. Structural OHLCV validation

In [3]:
structural_summary = pd.read_csv(
    ROOT / "reports/validation/structural_summary.csv"
)

structural_summary

,vendor,asset_id,nonpositive_price,negative_volume,low_above_body,high_below_body,high_below_low
0,yfinance,AAPL,0,0,0,0,0
1,yfinance,BRK.B,0,0,0,0,0
2,yfinance,COIN,0,0,0,0,0
3,yfinance,FIZZ,0,0,0,0,0
4,yfinance,KO,0,0,0,0,0
5,yfinance,MSFT,0,0,0,0,0
6,yfinance,NVDA,0,0,0,0,0
7,yfinance,PLTR,0,0,0,0,0
8,yfinance,SPY,0,0,0,0,0
9,yfinance,TSLA,0,0,0,0,0


As the table above shows, no asset from either vendor had nonpositive prices, negative volume, low above body, high below body or high below low. These checks identify structural impossibilities but do not prove every value is economically correct.

## 8. Market-calendar audit

In [4]:
calendar_summary = pd.read_csv(
    ROOT / "reports/validation/calendar_summary.csv"
)

calendar_summary

,asset_id,yfinance_missing_rows,yfinance_unexpected_sessions,alpha_vantage_missing_rows,alpha_vantage_unexpected_sessions
0,SPY,0,0,0,0
1,AAPL,0,0,0,0
2,MSFT,0,0,0,0
3,KO,0,0,0,0
4,NVDA,0,0,0,0
5,TSLA,0,0,0,0
6,FIZZ,0,0,0,0
7,PLTR,0,0,0,0
8,COIN,0,0,0,0
9,BRK.B,0,0,0,0


There were no missing expected sessions or unexpected sessions within the audited period. Automated calendar auditing begins on `1970-01-01`.
KO’s earlier observations are preserved but excluded from the automated gap test because the calendar package produced historical holiday false positives.

## 9. Partial-session detection

In [5]:
session_summary = pd.read_csv(
    ROOT / "reports/validation/session_completeness_summary.csv"
)

session_summary

,asset_id,yfinance_incomplete,alpha_vantage_incomplete
0,SPY,1,0
1,AAPL,1,0
2,MSFT,1,0
3,KO,1,0
4,NVDA,1,0
5,TSLA,1,0
6,FIZZ,1,0
7,PLTR,1,0
8,COIN,1,0
9,BRK.B,1,0


yfinance returned one potentially incomplete row per asset on `2026-07-17` because the snapshot was retrieved before the regular XNYS close. Those 10 rows remain in the interim panel for auditability but they are excluded from the finalized processed panel.
Alpha Vantage had no incomplete rows because its latest date was `2026-07-16`.

## 10. Suspicious-jump forensics

In [6]:
jump_summary = pd.read_csv(
    ROOT / "reports/validation/jump_summary.csv"
)

jump_summary

,asset_id,yfinance_jumps,alpha_vantage_jumps
0,SPY,0,0
1,AAPL,3,0
2,MSFT,1,0
3,KO,0,0
4,NVDA,10,0
5,TSLA,0,0
6,FIZZ,1,0
7,PLTR,2,0
8,COIN,2,0
9,BRK.B,0,0


In [7]:
jump_forensics = pd.read_parquet(
    ROOT / "reports/validation/jump_forensics.parquet"
)
jump_forensics.sort_values("abs_return", ascending=False,).head(20)
#There are only 19 flagged rows, so all will still appear

,,previous_close_raw,close_raw,raw_close_return,abs_return,split_factor,cash_dividend,source,vendor
date,asset_id,,,,,,,,
2000-09-29,AAPL,0.955357,0.459821,-0.518692,0.518692,1.0,0.0,yfinance,yfinance
2000-03-07,NVDA,0.121875,0.173568,0.424148,0.424148,1.0,0.0,yfinance,yfinance
2021-01-27,FIZZ,64.745003,90.754997,0.40173,0.40173,1.0,0.0,yfinance,yfinance
2004-08-06,NVDA,0.121333,0.078583,-0.352336,0.352336,1.0,0.0,yfinance,yfinance
1997-08-06,AAPL,0.176339,0.234933,0.33228,0.33228,1.0,0.0,yfinance,yfinance
2003-05-09,NVDA,0.133833,0.178083,0.330636,0.330636,1.0,0.0,yfinance,yfinance
2002-07-31,NVDA,0.135167,0.09225,-0.317511,0.317511,1.0,0.0,yfinance,yfinance
2024-11-06,COIN,193.960007,254.309998,0.311147,0.311147,1.0,0.0,yfinance,yfinance
2024-02-06,PLTR,16.719999,21.870001,0.308014,0.308014,1.0,0.0,yfinance,yfinance


As presented above, 19 yfinance observations exceeded the 25% threshold, with NVDA accounting for 10 of those. Alpha Vantage had no flagged jumps in its recent 100 observation window.
Some observations may be genuine market moves, while others may reflect historical vendor conventions or differences in the supplied price basis.
The split and dividend fields do not explain these rows automatically. The 25% threshold is therefore a diagnostic trigger for manual investigation, not a rule for labeling observations as errors.


## 11. Cross-vendor comparison

In [8]:
vendor_comparison_summary = pd.read_csv(
    ROOT / "reports/validation/vendor_comparison_summary.csv"
)

vendor_comparison = pd.read_parquet(
    ROOT / "data/processed/vendor_comparison.parquet"
)

print("Per-asset comparison summary")
vendor_comparison_summary

Per-asset comparison summary


,asset_id,overlap_rows,yfinance_only_rows,alpha_vantage_only_rows,median_close_rel_diff,max_close_rel_diff,median_volume_rel_diff,max_volume_rel_diff
0,SPY,100,8322,0,2.282695e-08,4.635802e-08,4.272863e-07,0.053539
1,AAPL,100,11389,0,2.722894e-08,5.658826e-08,5.261152e-07,0.024410
2,MSFT,100,10063,0,1.932985e-08,3.811124e-08,7.338764e-07,0.007494
3,KO,100,16142,0,2.383482e-08,4.824278e-08,1.863141e-06,0.031417
4,NVDA,100,6812,0,1.438664e-08,4.118895e-08,1.493247e-07,0.126543
5,TSLA,100,3936,0,2.208307e-08,1.249941e-05,4.268331e-07,0.018047
6,FIZZ,100,8671,0,2.368214e-08,1.397839e-04,8.364922e-05,0.009184
7,PLTR,100,1354,0,2.353409e-08,5.545710e-08,5.611575e-07,0.012048
8,COIN,100,1220,0,2.554822e-08,3.030709e-05,2.405779e-06,0.008932
9,BRK.B,100,7494,0,1.733395e-08,3.043136e-08,5.374036e-06,0.007216


In [9]:
overlap = vendor_comparison[vendor_comparison["yfinance_observed"] & vendor_comparison["alpha_vantage_observed"]].copy()

overlap["close_rel_diff_bps"] = overlap["close_rel_diff"] * 10_000

display_cols = [
    "yfinance_close_raw", 
    "alpha_vantage_close_raw",
    "close_abs_diff", 
    "close_rel_diff", 
    "close_rel_diff_bps",
    "yfinance_volume_raw", 
    "alpha_vantage_volume_raw"
]

largest_close_diffs = overlap.sort_values("close_rel_diff", ascending=False).head(15)
print("Largest close disagreements")
largest_close_diffs[display_cols]

Largest close disagreements


,,yfinance_close_raw,alpha_vantage_close_raw,close_abs_diff,close_rel_diff,close_rel_diff_bps,yfinance_volume_raw,alpha_vantage_volume_raw
date,asset_id,,,,,,,
2026-06-08,FIZZ,35.759998,35.755,0.004998,0.00014,1.397839,238800,238781
2026-06-17,COIN,164.919998,164.915,0.004998,0.00003,0.303071,8151500,8151519
2026-04-16,COIN,199.830002,199.825,0.005002,0.000025,0.250307,11235800,11235814
2026-03-10,TSLA,399.23999,399.235,0.00499,0.000012,0.124994,59258700,59258743
2026-04-06,AAPL,258.859985,258.86,0.000015,0.0,0.000566,29329900,29329911
2026-06-09,PLTR,132.070007,132.07,0.000007,0.0,0.000555,38680000,38679990
2026-07-02,FIZZ,33.330002,33.33,0.000002,0.0,0.000549,1373000,1373136
2026-04-27,AAPL,267.609985,267.61,0.000015,0.0,0.000547,41466800,41466762
2026-02-24,AAPL,272.140015,272.14,0.000015,0.0,0.000538,47014600,47014619


In [10]:
overlap["volume_signed_diff"] = overlap["yfinance_volume_raw"] - overlap["alpha_vantage_volume_raw"]
overlap["volume_abs_diff"] = overlap["volume_signed_diff"].abs()

overlap["yf_higher_vol"] = overlap["volume_signed_diff"] > 0
overlap["av_higher_vol"] = overlap["volume_signed_diff"] < 0


date_vol = overlap.reset_index().groupby("date").agg(
    num_assets=("asset_id", "count"),
    median_vol_rel_diff=("volume_rel_diff", "median"),
    max_vol_rel_diff=("volume_rel_diff", "max"),
    total_abs_vol_diff=("volume_abs_diff", "sum"),
    yf_higher_count=("yf_higher_vol", "sum"),
    av_higher_count=("av_higher_vol", "sum")
)

top_10_vol_dates = date_vol.sort_values("max_vol_rel_diff", ascending=False).head(10)
print("Date-level volume disagreements")
top_10_vol_dates

Date-level volume disagreements


,num_assets,median_vol_rel_diff,max_vol_rel_diff,total_abs_vol_diff,yf_higher_count,av_higher_count
date,,,,,,
2026-05-20,10,0.015048,0.126543,27249179,10,0
2026-07-16,10,0.002478,0.008273,1460817,0,10
2026-06-25,10,0.001652,0.004375,1412623,2,8
2026-07-02,10,0.001283,0.003907,888061,0,10
2026-06-12,10,0.001356,0.003062,694758,2,8
2026-03-24,10,0.0,0.002861,59893,4,6
2026-05-26,10,0.000001,0.002033,17158,4,6
2026-03-19,10,0.000001,0.001961,648,5,5
2026-05-28,10,0.000001,0.001809,7730,6,4


The tables above contain the cross-vendor comparison results. All assets have 100 overlapping observations, while Alpha Vantage has no additional dates beyond yfinance. Within these shared sessions, the closing prices reconcile extremely closely. The maximum close discrepancy in the entire sample is FIZZ on `2026-06-08`, at approximately 1.40 basis points.


Volume disagreements are larger and tend to cluster systematically by date. For instance, on 2026-05-20, yfinance reported higher volume for all 10 assets, generating a total absolute difference of about 27.25 million shares. This project does not declare either vendor authoritative because volume aggregation, correction, and historical-revision conventions may differ.

## 12. First and last observation review

In [11]:
def build_boundary_table(panel, vendor):

    first_rows = panel.groupby(level="asset_id", sort=True).head(1).reset_index()
    first_rows["boundary"] = "first"

    last_rows = panel.groupby(level="asset_id", sort=True).tail(1).reset_index()
    last_rows["boundary"] = "last"

    result = pd.concat([first_rows, last_rows], ignore_index=True,)
    result["vendor"] = vendor

    return result[
        [
            "vendor",
            "asset_id",
            "boundary",
            "date",
            "open_raw",
            "close_raw",
            "volume_raw",
            "adj_close",
            "cash_dividend",
            "split_factor",
            "retrieved_at",
        ]
    ]

In [12]:
boundary_table = pd.concat([build_boundary_table(yf_panel, "yfinance"),build_boundary_table(av_panel, "alpha_vantage"),],
    ignore_index=True,).sort_values(["asset_id", "vendor", "boundary"]).reset_index(drop=True)

boundary_table

,vendor,asset_id,boundary,date,open_raw,close_raw,volume_raw,adj_close,cash_dividend,split_factor,retrieved_at
0,alpha_vantage,AAPL,first,2026-02-23,263.49,266.18,37308155,<NA>,<NA>,<NA>,2026-07-17 17:05:51.011636+00:00
1,alpha_vantage,AAPL,last,2026-07-16,328.005,333.26,62970617,<NA>,<NA>,<NA>,2026-07-17 17:05:51.011636+00:00
2,yfinance,AAPL,first,1980-12-12,0.128348,0.128348,469033600,0.098207,0.0,1.0,2026-07-17 15:15:10.559532+00:00
3,yfinance,AAPL,last,2026-07-16,328.01001,333.26001,62727600,333.26001,0.0,1.0,2026-07-17 15:15:10.559532+00:00
4,alpha_vantage,BRK.B,first,2026-02-23,496.5,494.09,3556548,<NA>,<NA>,<NA>,2026-07-17 17:05:51.011636+00:00
5,alpha_vantage,BRK.B,last,2026-07-16,489.925,493.12,4194963,<NA>,<NA>,<NA>,2026-07-17 17:05:51.011636+00:00
6,yfinance,BRK.B,first,1996-05-09,22.200001,23.200001,4290000,23.200001,0.0,1.0,2026-07-17 15:15:10.559532+00:00
7,yfinance,BRK.B,last,2026-07-16,489.929993,493.119995,4187600,493.119995,0.0,1.0,2026-07-17 15:15:10.559532+00:00
8,alpha_vantage,COIN,first,2026-02-23,166.16,160.24,12685504,<NA>,<NA>,<NA>,2026-07-17 17:05:51.011636+00:00
9,alpha_vantage,COIN,last,2026-07-16,165.37,160.49,6328713,<NA>,<NA>,<NA>,2026-07-17 17:05:51.011636+00:00


Alpha Vantage begins on the same date for every asset because its free compact endpoint is limited to 100 observations, while yfinance start dates vary substantially by security. PLTR and COIN have shorter histories by design, dates before their first observations are therefore not treated as calendar gaps. The earliest yfinance date does not by itself prove that it is the exact IPO or first exchange-trading date. Both finalized panels end on `2026-07-16`. The potentially incomplete `2026-07-17` yfinance rows were preserved in interim storage but excluded from this table.

## 13. Unresolved ambiguities and limitations

- yfinance *_raw fields are unchanged by this project, but Yahoo may already supply them on a split-adjusted basis
- The free Alpha Vantage endpoint is limited to the 100 most recent observations, so it cannot alone validate the full yfinance history
- Vendor volume definitions, aggregation rules, corrections, or historical revisions may differ on particular dates
- The two vendor snapshots were retrieved approximately 1 hour and 51 minutes apart, so later database revisions could contribute to some differences
- Automated calendar auditing excludes observations before `1970-01-01` because the selected calendar package produced false positives for historical exchange holidays
- Suspicious jumps are flagged for investigation but are not conclusively classified as errors, corporate actions, or genuine market movements
- `asset_id` is project-controlled identifier rather than an institutional permanent security identifier
- Daily OHLCV cannot reconstruct spreads, order-book depth, venue, intraday path, or achievable execution prices
- Future downloads may differ because vendors can revise historical records

The possibility of silent historical vendor revisions is the most important limitation because it directly affects whether the same dataset can be reproduced in the future.

## 14. Conclusions

This project normalized daily data for 10 securities into one canonical schema with a unique and sorted (date, asset_id) index. Both vendor panels passed the structural OHLCV checks, with no nonpositive prices, negative volume, or invalid high–low relationships. Calendar auditing found no missing or unexpected sessions within the defined post-1970 audit period. Ten potentially incomplete yfinance rows dated `2026-07-17` were identified and excluded from the finalized processed panel while being preserved in interim storage. Nineteen unusually large yfinance price moves exceeded the 25% diagnostic threshold and were retained for further investigation rather than automatically repaired. Cross-vendor validation showed that daily closing prices reconciled very closely, while volume disagreements were larger and tended to cluster on specific dates. The resulting dataset is suitable for later return analysis, provided that its vendor conventions, coverage limitations, retrieval timestamps, and unresolved ambiguities remain attached to all further research.